In [ ]:
import json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix,
                             precision_recall_curve, average_precision_score, roc_auc_score)
from catboost import CatBoostClassifier, Pool
from IPython.display import display
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 100)

SEED = 42
DATA_PATH = "data/submissions.csv"
USE_CALENDAR_YEAR_MONTH = False   # set True for an ablation; year/month can be dataset-collection shortcuts
N_TRIALS = 30
TUNE_TIMEOUT_SEC = 3600
THRESH_MIN_PRECISION = None       # e.g. 0.85 -> best recall subject to anomaly precision >= 0.85; None -> maximize anomaly F1
MIN_PEER_N = 5                    # min rows in a peer group before its statistics are trusted

In [ ]:
def _clean_token(value):
    if pd.isna(value):
        return None
    token = " ".join(str(value).strip().split())
    return token if token else None

work_arrangement_aliases = {
    "hybrid": "hybrid", "onsite": "onsite", "on-site": "onsite", "on site": "onsite",
    "office": "onsite", "in-office": "onsite", "in office": "onsite",
    "remote": "remote", "remoteee": "remote", "wfh": "remote",
    "work from home": "remote", "work-from-home": "remote", "fully remote": "remote",
}
expected_work_arrangements = {"hybrid", "onsite", "remote"}

def normalize_work_arrangement(value):
    token = _clean_token(value)
    if token is None:
        return pd.NA
    token = token.lower()
    return work_arrangement_aliases.get(token, token)

valid_currency_codes = {"INR", "USD", "EUR", "GBP", "CAD", "SGD", "AUD", "PLN", "SEK", "CHF", "BRL", "AED"}
currency_aliases = {"INRR": "INR", "RS": "INR", "RS.": "INR", "RUPEE": "INR", "RUPEES": "INR", "US$": "USD"}
ambiguous_currency_aliases = {"RS", "RS."}
placeholder_currency_codes = {"XXX"}

def normalize_currency(value):
    token = _clean_token(value)
    if token is None:
        return pd.NA
    token = token.upper()
    if token in placeholder_currency_codes:
        return pd.NA
    if token in currency_aliases:
        return currency_aliases[token]
    if token in valid_currency_codes:
        return token
    return pd.NA

def classify_currency(value):
    token = _clean_token(value)
    if token is None:
        return "missing"
    raw = token
    token = token.upper()
    if token in placeholder_currency_codes:
        return "placeholder_code_review"
    if token in ambiguous_currency_aliases:
        return "ambiguous_alias_review"
    if token in currency_aliases:
        return "alias_normalized"
    if token in valid_currency_codes:
        return "standard" if raw == token else "alias_normalized"
    return "unrecognized_review"

def to_num(s):
    return pd.to_numeric(s.astype("string").str.replace(",", "", regex=False).str.strip(), errors="coerce")

def present(s):
    t = s.astype("string").str.strip()
    return t.notna() & ~t.isin(["", "[]", "nan", "None"])

def sdiv(a, b):
    r = a / b.where(b > 0)
    return r.replace([np.inf, -np.inf], np.nan)

def parse_offer_date(s):
    """'Mon Jul 10 2023 11:29:00 GMT+0000 (Coordinated Universal Time)' -> UTC timestamp."""
    s = s.astype("string")
    cleaned = s.str.replace(r"\s*\(.*\)\s*$", "", regex=True)
    out = pd.to_datetime(cleaned, format="%a %b %d %Y %H:%M:%S GMT%z", errors="coerce", utc=True)
    miss = out.isna() & s.notna()
    if miss.any():
        out[miss] = pd.to_datetime(cleaned[miss], errors="coerce", utc=True, format="mixed")
    return out

def add_strict_prior_count(frame, source_col, time_col, output_col):
    out = pd.Series(0, index=frame.index, dtype="int64")
    mask = frame[source_col].notna() & frame[time_col].notna()
    if mask.any():
        counts = (frame.loc[mask, [source_col, time_col]]
                  .groupby([source_col, time_col], dropna=True, sort=True)
                  .size().rename("event_count").reset_index()
                  .sort_values([source_col, time_col], kind="mergesort"))
        counts["prior_count"] = counts.groupby(source_col, sort=False)["event_count"].cumsum() - counts["event_count"]
        prior_map = counts.set_index([source_col, time_col])["prior_count"]
        row_keys = pd.MultiIndex.from_frame(frame.loc[mask, [source_col, time_col]])
        out.loc[mask] = prior_map.reindex(row_keys).to_numpy(dtype=np.int64)
    frame[output_col] = out

def add_behaviour_features(d):
    """Causal (past-only) behaviour features. Uses no labels, so it is safe to compute before the split."""
    d = d.copy()
    ents = [("userId", "user"), ("ipHash", "ip"), ("deviceHash", "device")]
    for key, name in ents:
        add_strict_prior_count(d, key, "submittedAt", f"{name}_prior_submission_count")
    s = d.sort_values("submittedAt", kind="mergesort", na_position="first")
    for key, name in ents:
        gap = s.groupby(key)["submittedAt"].diff().dt.total_seconds() / 3600.0
        d[f"{name}_hours_since_prev"] = gap.reindex(d.index)
    pairs = [("deviceHash", "userId", "device_distinct_users"), ("ipHash", "userId", "ip_distinct_users"),
             ("userId", "deviceHash", "user_distinct_devices"), ("userId", "ipHash", "user_distinct_ips"),
             ("userId", "company", "user_distinct_companies"), ("deviceHash", "company", "device_distinct_companies")]
    for key, val, name in pairs:
        m = s[key].notna() & s[val].notna()
        first = pd.Series(0, index=s.index)
        first.loc[m] = (~s.loc[m].duplicated([key, val])).astype(int)
        d[name] = first.groupby(s[key]).cumsum().reindex(d.index).fillna(0)
    return d

def _components(frame, cols):
    n = len(frame)
    rows, cidx, offset = [], [], n
    for c in cols:
        codes, uniques = pd.factorize(frame[c])
        m = codes >= 0
        rows.append(np.flatnonzero(m)); cidx.append(codes[m] + offset)
        offset += len(uniques)
    r, c_ = np.concatenate(rows), np.concatenate(cidx)
    g = coo_matrix((np.ones(len(r)), (r, c_)), shape=(offset, offset))
    _, labels = connected_components(g, directed=False)
    return labels[:n]

def build_groups(frame):
    """Group rows that share a user/device/IP so the same actor/ring never sits on both sides of a split."""
    for cols in (["userId", "deviceHash", "ipHash"], ["userId", "deviceHash"], ["userId"]):
        labels = _components(frame, cols)
        top = pd.Series(labels).value_counts(normalize=True).iloc[0]
        print(f"Grouping on {cols}: {labels.max() + 1} groups, largest group = {top:.1%} of rows")
        if top < 0.25:
            return labels
    return labels

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df["submittedAt"] = pd.to_datetime(df["submittedAt"], errors="coerce", utc=True, format="mixed")
df["offerDate"] = parse_offer_date(df["offerDate"])
print(f"submittedAt parsed: {df['submittedAt'].notna().mean():.1%} | offerDate parsed: {df['offerDate'].notna().mean():.1%}")

df = add_behaviour_features(df)
df = df[df["is_valid"].notna()].copy()
df["is_valid"] = df["is_valid"].astype(int)
print("Dataset:", df.shape, "| anomaly rate:", round((df["is_valid"] == 0).mean(), 4))

groups = build_groups(df)
y_all = df["is_valid"]

# 60 / 20 / 20, grouped + stratified
sg1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tv_pos, te_pos = next(sg1.split(df, y_all, groups))
sg2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(sg2.split(df.iloc[tv_pos], y_all.iloc[tv_pos], groups[tv_pos]))
tr_pos, va_pos = tv_pos[tr_rel], tv_pos[va_rel]

assert not (set(groups[tr_pos]) & set(groups[va_pos])), "group leakage train/val"
assert not (set(groups[tr_pos]) & set(groups[te_pos])), "group leakage train/test"
assert not (set(groups[va_pos]) & set(groups[te_pos])), "group leakage val/test"

X_tr_raw, X_va_raw, X_te_raw = (df.iloc[p].drop(columns=["is_valid"]) for p in (tr_pos, va_pos, te_pos))
y_tr_valid, y_va_valid, y_te_valid = (y_all.iloc[p] for p in (tr_pos, va_pos, te_pos))
y_tr, y_va, y_te = (1 - y_tr_valid), (1 - y_va_valid), (1 - y_te_valid)   # 1 = anomaly
groups_tr = groups[tr_pos]

for n, y_ in [("train", y_tr), ("val", y_va), ("test", y_te)]:
    print(f"{n}: {len(y_):>6} rows | anomaly rate {y_.mean():.4f}") 

In [ ]:
PRESENCE_COLS = [
    "yearsAtLevel", "salaryFormat", "employmentType", "salesComp", "negotiatedAmount", "gender", "ethnicity",
    "education", "otherDetails", "vestingSchedule", "tags", "additionalBonuses", "avgAnnualOptionsGranted",
    "totalOptionsGranted", "optionStrikePrice", "optionPreferredPrice", "totalOutstandingShares",
    "latestCompanyValuation", "companySize", "fundingStage", "annualTargetBonusValue",
    "annualTargetBonusPercentage", "userExchangeRate", "commentsAndRepliesCount", "threadId",  # threadId / comments flags: check for collection-process leakage
    "avgAnnualStockGrantValue", "avgAnnualBonusValue", "firstYearBonusValue", "totalStockGrantValue",
    "yearsAtCompany", "baseSalary", "totalCompensation",
]
KEEP_RAW_NUMERIC = [
    "yearsOfExperience", "yearsAtCompany", "yearsAtLevel", "salesComp", "negotiatedAmount",
    "avgAnnualOptionsGranted", "totalOptionsGranted", "optionStrikePrice", "optionPreferredPrice",
    "totalOutstandingShares", "latestCompanyValuation", "annualTargetBonusPercentage", "userExchangeRate",
    "commentsAndRepliesCount", "userAccountAgeDays", "userPriorSubmissions",
]
BEHAVIOUR_COLS = [
    "user_prior_submission_count", "ip_prior_submission_count", "device_prior_submission_count",
    "user_hours_since_prev", "ip_hours_since_prev", "device_hours_since_prev",
    "device_distinct_users", "ip_distinct_users", "user_distinct_devices", "user_distinct_ips",
    "user_distinct_companies", "device_distinct_companies",
]
RAW_CATEGORICAL = ["company", "jobFamily", "level", "focusTag", "location", "compPerspective", "gender", "education",
                   "salaryFormat", "employmentType", "ethnicity", "companySize", "fundingStage", "stockType"]

def _key(X, cols):
    k = X[cols[0]].astype(str)
    for c in cols[1:]:
        k = k + "|" + X[c].astype(str)
    return k

def fit_train_stats(X):
    """Label-free statistics fitted on TRAIN only."""
    st = {}
    cur = X["baseSalaryCurrency"].map(normalize_currency)
    r = pd.DataFrame({"c": cur.astype("string"), "r": to_num(X["exchangeRate"])}).dropna()
    r = r[r["r"] > 0]
    rate_map = r.groupby("c")["r"].median().to_dict()      # local currency units per 1 USD
    rate_map["USD"] = 1.0
    st["rate_map"] = rate_map
    ctry = X["locationSlug"].astype("string").str.extract(r"-([a-z]{3})$")[0]
    tmp = pd.DataFrame({"ctry": ctry, "c": cur.astype("string")}).dropna()
    st["country_cur"] = tmp.groupby("ctry")["c"].agg(lambda s: s.mode().iloc[0]).to_dict()
    st["freq"] = {
        "company": X["company"].astype(str).value_counts().to_dict(),
        "location": X["location"].astype(str).value_counts().to_dict(),
        "company_level": _key(X, ["company", "level"]).value_counts().to_dict(),
        "company_family_level": _key(X, ["company", "jobFamily", "level"]).value_counts().to_dict(),
    }
    print("USD conversion rates (local per USD):", {k: round(v, 4) for k, v in rate_map.items()})
    return st

def parse_vesting(v):
    if pd.isna(v) or str(v).strip() in ("", "[]"):
        return (np.nan, np.nan, np.nan, 0)
    try:
        lst = json.loads(v)
        pct = [float(t.get("percent", np.nan)) for t in lst]
        occ = [float(t.get("occurrences", np.nan)) for t in lst]
        return (np.nansum(pct), len(lst), np.nansum(occ), 0)
    except Exception:
        return (np.nan, np.nan, np.nan, 1)

def parse_addl_bonuses(v, rate_map):
    if pd.isna(v) or str(v).strip() in ("", "[]"):
        return (0, np.nan, 0)
    try:
        lst = json.loads(v)
        tot = 0.0
        for t in lst:
            rate = rate_map.get(str(t.get("currency", "")).upper(), np.nan)
            tot += float(t.get("value", 0)) / rate if rate == rate else np.nan
        return (len(lst), tot, 0)
    except Exception:
        return (np.nan, np.nan, 1)

def count_json_list(v):
    if pd.isna(v) or str(v).strip() in ("", "[]"):
        return 0
    try:
        return len(json.loads(v))
    except Exception:
        return np.nan

def base_features(X, st):
    F = pd.DataFrame(index=X.index)

    # ---- categoricals ----
    for c in RAW_CATEGORICAL:
        F[c] = X[c].astype("string").str.strip()
    F["workArrangement"] = X["workArrangement"].map(normalize_work_arrangement).astype("string")
    F["workArrangement_unexpected"] = (F["workArrangement"].notna() & ~F["workArrangement"].isin(expected_work_arrangements)).astype("int8")
    F["companyInfo_registered"] = X["companyInfo_registered"].astype("string")
    for c in ["cityId", "dmaId", "countryId"]:
        F[c] = to_num(X[c]).round().astype("Int64").astype("string")
    F["country_token"] = X["locationSlug"].astype("string").str.extract(r"-([a-z]{3})$")[0]
    for c in ["baseSalaryCurrency", "stockGrantCurrency", "bonusCurrency", "userCurrency"]:
        F[c] = X[c].map(normalize_currency).astype("string")
        F[f"{c}_status"] = X[c].map(classify_currency)

    # ---- numerics ----
    N = {c: to_num(X[c]) for c in
         ["baseSalary", "totalCompensation", "firstYearTotalCompensation", "avgAnnualStockGrantValue",
          "firstYearStockGrantValue", "totalStockGrantValue", "avgAnnualBonusValue", "firstYearBonusValue",
          "annualTargetBonusValue", "annualTargetBonusPercentage", "salesComp", "negotiatedAmount",
          "yearsOfExperience", "yearsAtCompany", "yearsAtLevel", "exchangeRate",
          "userPriorSubmissions", "userAccountAgeDays"] if c in X.columns}

    def rate_of(cur):
        return pd.to_numeric(cur.astype("string").map(st["rate_map"]), errors="coerce")
    r_base, r_stock, r_bonus = rate_of(F["baseSalaryCurrency"]), rate_of(F["stockGrantCurrency"]), rate_of(F["bonusCurrency"])

    # ---- unit conversion: everything to USD (rates fitted on train) ----
    usd = {
        "base": N["baseSalary"] / r_base,
        "total": N["totalCompensation"] / r_base,            # total comp is assumed to be quoted in the base currency
        "first_total": N["firstYearTotalCompensation"] / r_base,
        "avg_stock": N["avgAnnualStockGrantValue"] / r_stock,
        "first_stock": N["firstYearStockGrantValue"] / r_stock,
        "total_stock": N["totalStockGrantValue"] / r_stock,
        "avg_bonus": N["avgAnnualBonusValue"] / r_bonus,
        "first_bonus": N["firstYearBonusValue"] / r_bonus,
        "target_bonus": N["annualTargetBonusValue"] / r_bonus,
        "sales": N["salesComp"] / r_base,
        "negotiated": N["negotiatedAmount"] / r_base,
    }
    for k, v in usd.items():
        F[f"{k}_usd_log"] = np.log1p(v.clip(lower=0))
    F["baseSalary_local"] = N["baseSalary"]                  # keep native scale: e.g. INR 20,900 is itself suspicious
    F["totalCompensation_local"] = N["totalCompensation"]
    F["exchangeRate_dev"] = np.log(N["exchangeRate"].where(N["exchangeRate"] > 0) / r_base)
    F["currency_mismatch_base_stock"] = (F["baseSalaryCurrency"] != F["stockGrantCurrency"]).where(F["baseSalaryCurrency"].notna() & F["stockGrantCurrency"].notna()).astype(float)
    F["currency_mismatch_base_bonus"] = (F["baseSalaryCurrency"] != F["bonusCurrency"]).where(F["baseSalaryCurrency"].notna() & F["bonusCurrency"].notna()).astype(float)

    # ---- accounting identities / residuals / ratios ----
    sum_raw = N["baseSalary"].fillna(0) + N["avgAnnualStockGrantValue"].fillna(0) + N["avgAnnualBonusValue"].fillna(0)
    F["tc_resid_raw"] = N["totalCompensation"] - sum_raw
    F["tc_resid_raw_rel"] = sdiv(F["tc_resid_raw"], N["totalCompensation"])
    first_sum_raw = N["baseSalary"].fillna(0) + N["firstYearStockGrantValue"].fillna(0) + N["firstYearBonusValue"].fillna(0)
    F["ftc_resid_raw"] = N["firstYearTotalCompensation"] - first_sum_raw
    F["ftc_resid_raw_rel"] = sdiv(F["ftc_resid_raw"], N["firstYearTotalCompensation"])
    sum_usd = usd["base"].fillna(0) + usd["avg_stock"].fillna(0) + usd["avg_bonus"].fillna(0)
    F["tc_resid_usd_rel"] = sdiv(usd["total"] - sum_usd, usd["total"])
    F["stock_implied_vest_years"] = sdiv(N["totalStockGrantValue"], N["avgAnnualStockGrantValue"])
    F["bonus_pct_implied"] = sdiv(usd["target_bonus"], usd["base"]) * 100
    F["bonus_pct_resid"] = N["annualTargetBonusPercentage"] - F["bonus_pct_implied"]
    F["ftc_over_tc"] = sdiv(N["firstYearTotalCompensation"], N["totalCompensation"])
    F["stock_to_base"] = sdiv(usd["avg_stock"], usd["base"])
    F["bonus_to_base"] = sdiv(usd["avg_bonus"], usd["base"])
    F["total_to_base"] = sdiv(usd["total"], usd["base"])
    F["first_stock_vs_avg"] = sdiv(N["firstYearStockGrantValue"], N["avgAnnualStockGrantValue"])
    F["first_bonus_vs_avg"] = sdiv(N["firstYearBonusValue"], N["avgAnnualBonusValue"])
    F["base_negative_or_zero"] = (N["baseSalary"] <= 0).where(N["baseSalary"].notna()).astype(float)

    # ---- experience consistency ----
    yoe, yac, yal = N["yearsOfExperience"], N["yearsAtCompany"], N["yearsAtLevel"]
    F["yac_gt_yoe"] = (yac > yoe).where(yac.notna() & yoe.notna()).astype(float)
    F["yal_gt_yac"] = (yal > yac).where(yal.notna() & yac.notna()).astype(float)
    F["yoe_minus_yac"] = yoe - yac
    F["yoe_out_of_range"] = ((yoe < 0) | (yoe > 50)).where(yoe.notna()).astype(float)

    # ---- time features / logical date consistency ----
    sub, off = X["submittedAt"], X["offerDate"]
    both = sub.notna() & off.notna()
    F["days_since_offer"] = (sub - off).dt.total_seconds() / 86400.0
    F["offer_after_submit"] = (off > sub).where(both).astype(float)
    F["offer_same_time_of_day"] = ((sub.dt.hour * 60 + sub.dt.minute) == (off.dt.hour * 60 + off.dt.minute)).where(both).astype(float)
    F["submit_hour"] = sub.dt.hour
    F["submit_dayofweek"] = sub.dt.dayofweek
    if USE_CALENDAR_YEAR_MONTH:
        F["submit_year"], F["submit_month"] = sub.dt.year, sub.dt.month

    # ---- identity / naming consistency ----
    F["company_name_mismatch"] = (X["company"].astype(str).str.lower().str.strip() != X["companyInfo_name"].astype(str).str.lower().str.strip()).where(X["company"].notna() & X["companyInfo_name"].notna()).astype(float)
    F["company_info_unknown"] = (X["companyInfo_name"].astype(str).str.lower() == "unknown").astype(float)
    F["title_eq_jobFamily"] = (X["title"].astype(str).str.lower().str.strip() == X["jobFamily"].astype(str).str.lower().str.strip()).where(X["title"].notna() & X["jobFamily"].notna()).astype(float)
    F["title_len"] = X["title"].astype("string").str.len()
    F["otherDetails_len"] = X["otherDetails"].astype("string").str.len()
    exp_cur = F["country_token"].map(st["country_cur"])
    F["currency_country_mismatch"] = (F["baseSalaryCurrency"] != exp_cur).where(F["baseSalaryCurrency"].notna() & exp_cur.notna()).astype(float)

    # ---- frequency encodings (rare company/level combos are suspicious) ----
    F["company_freq"] = np.log1p(X["company"].astype(str).map(st["freq"]["company"]).fillna(0))
    F["location_freq"] = np.log1p(X["location"].astype(str).map(st["freq"]["location"]).fillna(0))
    F["company_level_freq"] = np.log1p(_key(X, ["company", "level"]).map(st["freq"]["company_level"]).fillna(0))
    F["company_family_level_freq"] = np.log1p(_key(X, ["company", "jobFamily", "level"]).map(st["freq"]["company_family_level"]).fillna(0))

    # ---- structured text -> numbers ----
    v = pd.DataFrame(X["vestingSchedule"].map(parse_vesting).tolist(), index=X.index,
                     columns=["vest_pct_sum", "vest_n_tranches", "vest_total_occ", "vest_malformed"])
    F = F.join(v)
    F["vest_pct_not_100"] = ((F["vest_pct_sum"] - 100).abs() > 1).where(F["vest_pct_sum"].notna()).astype(float)
    a = pd.DataFrame(X["additionalBonuses"].map(lambda z: parse_addl_bonuses(z, st["rate_map"])).tolist(), index=X.index,
                     columns=["addl_bonus_n", "addl_bonus_usd", "addl_bonus_malformed"])
    F = F.join(a)
    F["tags_count"] = X["tags"].map(count_json_list)

    # ---- is-present flags (replace dropping >80%-missing columns) ----
    for c in PRESENCE_COLS:
        if c in X.columns:
            F[f"has_{c}"] = present(X[c]).astype("int8")
    for c in KEEP_RAW_NUMERIC:
        if c in X.columns:
            F[c] = N[c] if c in N else to_num(X[c])

    # ---- behaviour / graph (causal) features ----
    for c in BEHAVIOUR_COLS:
        if c in X.columns:
            F[c] = X[c]
    F["prior_declared_minus_observed"] = N["userPriorSubmissions"] - X["user_prior_submission_count"]
    return F

st = fit_train_stats(X_tr_raw)
F_tr, F_va, F_te = (base_features(x, st) for x in (X_tr_raw, X_va_raw, X_te_raw))
print("Base feature matrices:", F_tr.shape, F_va.shape, F_te.shape)

In [ ]:
CHECK_FLAGS = ["offer_after_submit", "offer_same_time_of_day", "yac_gt_yoe", "yal_gt_yac", "yoe_out_of_range",
               "company_name_mismatch", "company_info_unknown", "title_eq_jobFamily", "currency_country_mismatch",
               "vest_pct_not_100", "vest_malformed", "addl_bonus_malformed", "base_negative_or_zero",
               "workArrangement_unexpected", "currency_mismatch_base_stock", "currency_mismatch_base_bonus"]
rows = []
for c in CHECK_FLAGS:
    if c in F_tr.columns:
        m = (F_tr[c] == 1)
        rows.append({"check": c, "n_flagged": int(m.sum()), "pct_rows": round(100 * m.mean(), 2),
                     "anomaly_rate_if_flagged": round(y_tr[m].mean(), 3) if m.any() else np.nan,
                     "anomaly_rate_if_not": round(y_tr[~m].mean(), 3)})
print(f"Baseline anomaly rate (train): {y_tr.mean():.3f}")
display(pd.DataFrame(rows).sort_values("anomaly_rate_if_flagged", ascending=False))

print("\nIdentity residuals by label (train), quantiles of |relative residual|:")
resid_cols = ["tc_resid_raw_rel", "ftc_resid_raw_rel", "tc_resid_usd_rel", "bonus_pct_resid", "exchangeRate_dev"]
display(F_tr[resid_cols].abs().groupby(y_tr.rename("is_anomaly")).quantile([.5, .9, .99]).round(4))

In [ ]:
PEER_LEVELS = {
    "cjll": ["company", "jobFamily", "level", "location"],
    "cjl":  ["company", "jobFamily", "level"],
    "jll":  ["jobFamily", "level", "location"],
    "cj":   ["company", "jobFamily"],
    "jl":   ["jobFamily", "level"],
    "j":    ["jobFamily"],
}
PEER_TARGETS = ["base_usd_log", "total_usd_log"]

def fit_peer_stats(F, valid_mask):
    G = F.loc[valid_mask]                        # "normal" behaviour learned from VALID rows only
    stats = {}
    for name, keys in PEER_LEVELS.items():
        for t in PEER_TARGETS:
            sub = G.loc[G[t] > 0, keys + [t]].dropna()
            g = sub.groupby(keys, observed=True)[t]
            tab = pd.DataFrame({"med": g.median(), "q25": g.quantile(.25), "q75": g.quantile(.75), "n": g.size()})
            tab["scale"] = ((tab["q75"] - tab["q25"]) / 1.349).clip(lower=0.05)
            stats[(name, t)] = tab[["med", "scale", "n"]]
    return stats

def _lookup(F, keys, tab):
    t_ = tab[tab["n"] >= MIN_PEER_N].reset_index()
    j = F[keys].reset_index(drop=True).merge(t_, on=keys, how="left")
    return j["med"].to_numpy(float), j["scale"].to_numpy(float), j["n"].to_numpy(float)

def apply_peer_stats(F, stats):
    out = pd.DataFrame(index=F.index)
    for t in PEER_TARGETS:
        actual = F[t].where(F[t] > 0).to_numpy(float)
        exp, scale, n, lvl = (np.full(len(F), np.nan) for _ in range(4))
        looked = {}
        for i, (name, keys) in enumerate(PEER_LEVELS.items()):
            med, sc, nn = _lookup(F, keys, stats[(name, t)])
            looked[name] = med
            take = np.isnan(exp) & ~np.isnan(med)           # most specific level with enough peers wins
            exp[take], scale[take], n[take], lvl[take] = med[take], sc[take], nn[take], i
        resid = actual - exp
        out[f"peer_exp_{t}"] = exp
        out[f"peer_resid_{t}"] = resid
        out[f"peer_z_{t}"] = resid / scale
        out[f"peer_n_{t}"] = n
        out[f"peer_lvl_{t}"] = lvl
        # cross-location expectation: same company/role/level elsewhere + this location's role/level premium
        xloc = looked["cjl"] + (looked["jll"] - looked["jl"])
        out[f"peer_xloc_resid_{t}"] = actual - xloc
    return out

def peer_features_oof(F, y_valid, groups, n_splits=5):
    parts = []
    for fit_i, app_i in GroupKFold(n_splits=n_splits).split(F, groups=groups):
        stats = fit_peer_stats(F.iloc[fit_i], (y_valid.iloc[fit_i] == 1).to_numpy())
        parts.append(apply_peer_stats(F.iloc[app_i], stats))
    return pd.concat(parts).reindex(F.index)

# Train: out-of-fold. Val/test: fitted on all train-valid rows.
P_tr = peer_features_oof(F_tr, y_tr_valid, groups_tr)
stats_full = fit_peer_stats(F_tr, (y_tr_valid == 1).to_numpy())
P_va, P_te = apply_peer_stats(F_va, stats_full), apply_peer_stats(F_te, stats_full)

X_tr, X_va, X_te = F_tr.join(P_tr), F_va.join(P_va), F_te.join(P_te)

# ---- final cleaning (all decisions fitted on train) ----
X_va, X_te = X_va.reindex(columns=X_tr.columns), X_te.reindex(columns=X_tr.columns)
const_cols = [c for c in X_tr.columns if X_tr[c].nunique(dropna=False) <= 1]
print("Dropping constant columns:", const_cols)
X_tr, X_va, X_te = (x.drop(columns=const_cols) for x in (X_tr, X_va, X_te))

cat_cols = X_tr.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
num_cols = [c for c in X_tr.columns if c not in cat_cols]
for x in (X_tr, X_va, X_te):
    for c in cat_cols:
        x[c] = x[c].astype("string").fillna("__MISSING__").astype(str)
    for c in num_cols:                                   # NaNs are kept: CatBoost handles them natively
        x[c] = pd.to_numeric(x[c], errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float32")

print(f"Final -> train {X_tr.shape}, val {X_va.shape}, test {X_te.shape} | {len(cat_cols)} categorical, {len(num_cols)} numeric")
pool_tr = Pool(X_tr, y_tr, cat_features=cat_cols)
pool_va = Pool(X_va, y_va, cat_features=cat_cols)
pool_te = Pool(X_te, y_te, cat_features=cat_cols)

In [ ]:
def make_params(trial):
    p = dict(
        iterations=3000,
        learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        depth=trial.suggest_int("depth", 4, 9),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 30, log=True),
        random_strength=trial.suggest_float("random_strength", 0.1, 10, log=True),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0, 5),
        border_count=trial.suggest_categorical("border_count", [64, 128, 254]),
        one_hot_max_size=trial.suggest_categorical("one_hot_max_size", [2, 10, 25]),
    )
    acw = trial.suggest_categorical("auto_class_weights", ["None", "SqrtBalanced", "Balanced"])
    if acw != "None":
        p["auto_class_weights"] = acw
    return p

def objective(trial):
    m = CatBoostClassifier(**make_params(trial), loss_function="Logloss", eval_metric="PRAUC",
                           early_stopping_rounds=150, random_seed=SEED, thread_count=-1,
                           allow_writing_files=False, verbose=0)
    m.fit(pool_tr, eval_set=pool_va, use_best_model=True)
    return m.get_best_score()["validation"]["PRAUC"]

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, timeout=TUNE_TIMEOUT_SEC, show_progress_bar=True)
print(f"Best validation PR-AUC (anomaly class): {study.best_value:.4f}")
print("Best params:", study.best_params)

In [ ]:
best = dict(study.best_params)
if best.get("auto_class_weights") == "None":
    best.pop("auto_class_weights")

final_model = CatBoostClassifier(**best, iterations=5000, loss_function="Logloss", eval_metric="PRAUC",
                                 early_stopping_rounds=200, random_seed=SEED, thread_count=-1,
                                 allow_writing_files=False, verbose=200)
final_model.fit(pool_tr, eval_set=pool_va, use_best_model=True)
print("Best iteration:", final_model.get_best_iteration())

def pick_threshold(y, p, min_precision=None):
    prec, rec, thr = precision_recall_curve(y, p)
    prec, rec = prec[:-1], rec[:-1]
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    if min_precision is not None:
        ok = prec >= min_precision
        if ok.any():
            return float(thr[ok][np.argmax(rec[ok])])
    return float(thr[np.argmax(f1)])

p_tr = final_model.predict_proba(pool_tr)[:, 1]
p_va = final_model.predict_proba(pool_va)[:, 1]
p_te = final_model.predict_proba(pool_te)[:, 1]

THRESHOLD = pick_threshold(y_va.to_numpy(), p_va, THRESH_MIN_PRECISION)   # chosen on validation only
print(f"\nChosen anomaly-probability threshold (from validation): {THRESHOLD:.4f}")

def report(name, y_anom, p, thr):
    pred_anom = (p >= thr).astype(int)
    y_v, pred_v = 1 - y_anom.to_numpy(), 1 - pred_anom      # back to your is_valid convention
    print(f"\n{'=' * 18} {name} METRICS {'=' * 18}")
    print(f"Accuracy:                 {accuracy_score(y_v, pred_v):.4f}")
    print(f"Macro F1:                 {f1_score(y_v, pred_v, average='macro'):.4f}")
    print(f"F1 (is_valid=0, anomaly): {f1_score(y_v, pred_v, pos_label=0):.4f}")
    print(f"F1 (is_valid=1):          {f1_score(y_v, pred_v, pos_label=1):.4f}")
    print(f"PR-AUC (anomaly):         {average_precision_score(y_anom, p):.4f}")
    print(f"ROC-AUC:                  {roc_auc_score(y_anom, p):.4f}")
    print("\nClassification report:")
    print(classification_report(y_v, pred_v, labels=[0, 1], target_names=["is_valid=0", "is_valid=1"], digits=4, zero_division=0))
    print("Confusion matrix (rows=true [0, 1], columns=predicted [0, 1]):")
    print(confusion_matrix(y_v, pred_v, labels=[0, 1]))

report("TRAIN", y_tr, p_tr, THRESHOLD)
report("VALIDATION", y_va, p_va, THRESHOLD)
report("TEST", y_te, p_te, THRESHOLD)

In [ ]:
imp = pd.Series(final_model.get_feature_importance(pool_tr), index=X_tr.columns).sort_values(ascending=False)
print("Top 30 features:"); display(imp.head(30).round(3).to_frame("importance"))
print("Features with ~zero importance (candidates to drop):", imp[imp < 0.01].index.tolist())

fn_idx = X_te.index[(y_te.to_numpy() == 1) & (p_te < THRESHOLD)]
print(f"\nMissed anomalies on test: {len(fn_idx)} of {int(y_te.sum())}")
cols = ["company", "level", "jobFamily", "location", "baseSalary", "baseSalaryCurrency", "totalCompensation",
        "yearsOfExperience", "yearsAtCompany", "offerDate", "submittedAt"]
display(df.loc[fn_idx, [c for c in cols if c in df.columns]].head(25))